# Fase 01b: Control de Calidad — Quarantine & Failed
**Objetivo:** Clasificar registros de la capa Bronze en 3 categorías según reglas de validación: failed, quarantine y clean.
**Entradas:** Parquet desde `medicalproyect-raw/bronze/`.
**Salidas:** Clean (continúa en Bronze), failed en `medicalproyect-processed/rejected/`, quarantine en `medicalproyect-processed/quarantine/` + logs de calidad.


## 1. Inicialización de Spark y Logger


In [1]:
def _subir_log(sufijo, storage_account):
    try:
        from notebookutils import mssparkutils
        import os
        from datetime import datetime
        fecha = datetime.now().strftime('%Y%m%d_%H%M%S')
        ruta_local = "file://" + os.path.abspath(log_filename)
        ruta_destino = f"abfss://medicalproyect-logs@{storage_account}.dfs.core.windows.net/{sufijo}.log"
        mssparkutils.fs.cp(ruta_local, ruta_destino)
    except Exception:
        pass


# ── Función helper para notificaciones por Telegram ──
def _enviar_telegram(mensaje):
    try:
        from notebookutils import mssparkutils
        import requests
        token = mssparkutils.credentials.getSecret(NOMBRE_BOVEDA, "TelegramToken", NOMBRE_PUENTE)
        chat_id = mssparkutils.credentials.getSecret(NOMBRE_BOVEDA, "TelegramChatId", NOMBRE_PUENTE)
        url = f"https://api.telegram.org/bot{token}/sendMessage"
        requests.post(url, json={"chat_id": chat_id, "text": mensaje, "parse_mode": "HTML"}, timeout=10)
    except Exception:
        pass

import logging
from datetime import datetime
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, lit, current_date, to_date, regexp_extract, concat_ws
)

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

log_filename = "pipeline_calidad.log"
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-7s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.FileHandler(log_filename, mode='a', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("Medical.Calidad")

logger.info("=" * 60)
logger.info("INICIO PIPELINE DE CALIDAD — QUARANTINE & FAILED")
logger.info("=" * 60)

STORAGE_ACCOUNT = "stproyectomastergrupo3"
CONTAINER_RAW       = "medicalproyect-raw"

# ── Configuración de Key Vault ──
NOMBRE_BOVEDA = "kv-medicalproyect"
NOMBRE_PUENTE = "AzureKeyVault"

CONTAINER_PROCESSED = "medicalproyect-processed"
CONTAINER_LOGS      = "medicalproyect-logs"
base_bronze     = f"abfss://{CONTAINER_RAW}@{STORAGE_ACCOUNT}.dfs.core.windows.net/bronze"
base_quarantine = f"abfss://{CONTAINER_PROCESSED}@{STORAGE_ACCOUNT}.dfs.core.windows.net/quarantine"
base_rejected   = f"abfss://{CONTAINER_PROCESSED}@{STORAGE_ACCOUNT}.dfs.core.windows.net/rejected"

spark = SparkSession.builder.appName("Medical_Calidad").getOrCreate()

df_patients    = spark.read.parquet(f"{base_bronze}/patients")
df_outcomes    = spark.read.parquet(f"{base_bronze}/outcomes")
df_medications = spark.read.parquet(f"{base_bronze}/medications")
df_lab_results = spark.read.parquet(f"{base_bronze}/lab_results")
df_diagnoses   = spark.read.parquet(f"{base_bronze}/diagnoses")

logger.info("Datos Bronze cargados.")

logger.info("📱 Enviando notificación de inicio...")
_enviar_telegram("🚀 INICIADO: 01B Calidad Bronze")


## 2. Función Helper: Clasificación Failed / Quarantine / Clean


In [ ]:
# ══════════════════════════════════════════════════════════════
# HELPER: separar failed / quarantine / clean y guardar
# ══════════════════════════════════════════════════════════════

# ── FUNCIÓN PRINCIPAL: separa registros en failed, quarantine y clean según reglas ──
def aplicar_reglas(df, reglas_failed, reglas_quarantine, nombre_tabla):
    """
    reglas_failed/quarantine: lista de tuplas (condicion_col, motivo_str)
    Devuelve (df_clean, resumen_dict)
    """
    total = df.count()

    # ── FAILED: inválidos bloqueantes ──

    # ── 1) FAILED: identificamos registros con errores bloqueantes ──
    cond_failed = lit(False)
    motivos_failed = lit(None).cast("string")
    for cond, motivo in reglas_failed:
        cond_failed = cond_failed | cond
        motivos_failed = when(cond, when(motivos_failed.isNull(), lit(motivo))
                                   .otherwise(concat_ws("|", motivos_failed, lit(motivo)))) \
                     .otherwise(motivos_failed)

    df_failed = df.filter(cond_failed) \
                  .withColumn("motivo_rechazo", motivos_failed) \
                  .withColumn("tabla_origen",   lit(nombre_tabla)) \
                  .withColumn("ts_rechazo",     lit(datetime.now().strftime('%Y-%m-%d %H:%M:%S')))

    n_failed = df_failed.count()

    # ── QUARANTINE: sospechosos pero no bloqueantes ──

    # ── 2) QUARANTINE: sobre los no fallidos, aplicamos reglas sospechosas ──
    df_no_failed = df.filter(~cond_failed)

    cond_quar  = lit(False)
    motivos_q = lit(None).cast("string")
    for cond, motivo in reglas_quarantine:
        cond_quar = cond_quar | cond
        motivos_q = when(cond, when(motivos_q.isNull(), lit(motivo))
                               .otherwise(concat_ws("|", motivos_q, lit(motivo)))) \
                     .otherwise(motivos_q)

    df_quarantine = df_no_failed.filter(cond_quar) \
                                .withColumn("motivo_alerta",  motivos_q) \
                                .withColumn("tabla_origen",   lit(nombre_tabla)) \
                                .withColumn("ts_alerta",      lit(datetime.now().strftime('%Y-%m-%d %H:%M:%S')))

    n_quarantine = df_quarantine.count()

    # ── CLEAN: pasa todas las validaciones ──

    # ── 3) CLEAN: los que pasan todas las reglas continúan al Silver ──
    df_clean = df_no_failed.filter(~cond_quar)
    n_clean  = df_clean.count()

    # ── Persistencia ──

    # ── 4) Guardamos failed y quarantine en el Data Lake para auditoría ──
    if n_failed > 0:
        df_failed.write.mode("overwrite").parquet(f"{base_rejected}/{nombre_tabla}")
    if n_quarantine > 0:
        df_quarantine.write.mode("overwrite").parquet(f"{base_quarantine}/{nombre_tabla}")
    # Log del desglose por motivo para saber qué reglas se activaron
    if n_failed > 0:
        logger.info(f"    ── Motivos FAILED ({nombre_tabla}) ──")
        rows = df_failed.groupBy("motivo_rechazo").count().orderBy("count", ascending=False).collect()
        for row in rows:
            logger.info(f"      - {row['motivo_rechazo']}: {row['count']} registros")
    if n_quarantine > 0:
        logger.info(f"    ── Motivos QUARANTINE ({nombre_tabla}) ──")
        rows = df_quarantine.groupBy("motivo_alerta").count().orderBy("count", ascending=False).collect()
        for row in rows:
            logger.info(f"      - {row['motivo_alerta']}: {row['count']} registros")
        

    # ── 5) Devolvemos el DataFrame limpio y el resumen de cuentas ──
    resumen = {
        "tabla":       nombre_tabla,
        "total":       total,
        "failed":      n_failed,
        "quarantine":  n_quarantine,
        "clean":       n_clean,
        "pct_clean":   round(n_clean / total * 100, 2) if total > 0 else 0
    }

    logger.info(f"  [{nombre_tabla}] total={total} | failed={n_failed} | "
                f"quarantine={n_quarantine} | clean={n_clean} ({resumen['pct_clean']}%)")

    return df_clean, resumen

resumenes = []
# Subida parcial de log tras esta seccion
_subir_log("calidad/calidad_validación_de_tablas", STORAGE_ACCOUNT)


## 3. Validación de Tablas


In [ ]:
# ══════════════════════════════════════════════════════════════
# PATIENTS
# ══════════════════════════════════════════════════════════════
logger.info("")
logger.info("── Validando PATIENTS ──")

# ── VALIDACIÓN DE PATIENTS ──
# Reglas FAILED: patient_id nulo, edad inválida, presión diastólica >= sistólica
# (datos imposibles fisiológicamente → no pueden procesarse)
failed_patients = [
    (col("patient_id").isNull(),                          "patient_id_nulo"),
    (col("age").isNull() | (col("age") < 0) | (col("age") > 120),
                                                          "age_invalida"),
    (col("diastolic_bp") >= col("systolic_bp"),           "bp_diastolica_mayor_sistolica"),
]

# Reglas QUARANTINE: valores biométricos extremos pero no imposibles
# (BMI, presión sistólica/diastólica, pulso, temperatura, Charlson)
quarantine_patients = [
    ((col("bmi") < 10) | (col("bmi") > 80),              "bmi_fuera_rango"),
    ((col("systolic_bp") < 50) | (col("systolic_bp") > 250),
                                                          "systolic_bp_extrema"),
    ((col("diastolic_bp") < 30) | (col("diastolic_bp") > 150),
                                                          "diastolic_bp_extrema"),
    ((col("heart_rate") < 20) | (col("heart_rate") > 250),
                                                          "heart_rate_extrema"),
    ((col("temperature_f") < 86) | (col("temperature_f") > 113),
                                                          "temperatura_extrema"),
    (col("charlson_index") < 0,                           "charlson_negativo"),
]

# Aplicamos las reglas a patients
df_patients_clean, r = aplicar_reglas(
    df_patients, failed_patients, quarantine_patients, "patients"
)
resumenes.append(r)

In [ ]:
# ══════════════════════════════════════════════════════════════
# OUTCOMES
# ══════════════════════════════════════════════════════════════
logger.info("")
logger.info("── Validando OUTCOMES ──")

# ── VALIDACIÓN DE OUTCOMES ──
# Reglas FAILED: patient_id nulo, fechas inconsistentes, estancia negativa, readmisión inválida
failed_outcomes = [
    (col("patient_id").isNull(),
        "patient_id_nulo"),
    (col("discharge_date") < col("admission_date"),
        "discharge_antes_admission"),
    (col("length_of_stay_days") < 0,
        "estancia_negativa"),
    ((col("readmitted_30d") == 1) & (col("days_to_readmission") > 30),
        "readmitted_inconsistente"),
    (col("days_to_readmission") < 0,
        "dias_readmision_negativos"),          # movido de quarantine a failed
]

# Reglas QUARANTINE: cargos negativos, estancia >1 año, ICU > estancia total
quarantine_outcomes = [
    (col("total_charges_usd") < 0,
        "cargos_negativos"),
    (col("length_of_stay_days") > 365,
        "estancia_superior_1anio"),
    (col("icu_days") > col("length_of_stay_days"),
        "icu_days_mayor_estancia"),
]

# Aplicamos las reglas a outcomes
df_outcomes_clean, r = aplicar_reglas(
    df_outcomes, failed_outcomes, quarantine_outcomes, "outcomes"
)
resumenes.append(r)

In [ ]:
# ══════════════════════════════════════════════════════════════
# LAB RESULTS
# ══════════════════════════════════════════════════════════════
logger.info("")
logger.info("── Validando LAB_RESULTS ──")

# ── VALIDACIÓN DE LAB_RESULTS ──
# Reglas FAILED: patient_id nulo, fecha futura, rango de referencia invertido
failed_lab = [
    (col("patient_id").isNull(),                        "patient_id_nulo"),
    (col("test_date") > current_date(),                 "fecha_futura"),
    (col("reference_high") < col("reference_low"),      "rango_referencia_invertido"),
]

# Reglas QUARANTINE: valor negativo o flag abnormal inconsistente
quarantine_lab = [
    (col("value") < 0,                                  "valor_analitica_negativo"),
    (
        (col("is_abnormal") == 0) &
        ((col("value") < col("reference_low")) | (col("value") > col("reference_high"))),
        "is_abnormal_inconsistente"
    ),
]

# Aplicamos las reglas a lab_results
df_lab_clean, r = aplicar_reglas(
    df_lab_results, failed_lab, quarantine_lab, "lab_results"
)
resumenes.append(r)

In [ ]:
# ══════════════════════════════════════════════════════════════
# MEDICATIONS
# ══════════════════════════════════════════════════════════════
logger.info("")
logger.info("── Validando MEDICATIONS ──")

# ── VALIDACIÓN DE MEDICATIONS ──
# Reglas FAILED: patient_id nulo, dosis cero o negativa
failed_med = [
    (col("patient_id").isNull(),                        "patient_id_nulo"),
    ((col("dose") <= 0),                                "dosis_cero_o_negativa"),
]

# Reglas QUARANTINE: fecha futura, duración negativa, adherencia fuera de 0-100%
quarantine_med = [
    (col("start_date") > current_date(),                "fecha_inicio_futura"),
    (col("duration_days") < 0,                          "duracion_negativa"),
    ((col("adherence_pct") < 0) | (col("adherence_pct") > 100),
                                                        "adherencia_fuera_rango"),
]

# Aplicamos las reglas a medications
df_med_clean, r = aplicar_reglas(
    df_medications, failed_med, quarantine_med, "medications"
)
resumenes.append(r)

In [ ]:
# ══════════════════════════════════════════════════════════════
# DIAGNOSES
# ══════════════════════════════════════════════════════════════
logger.info("")
logger.info("── Validando DIAGNOSES ──")

# ── VALIDACIÓN DE DIAGNOSES ──
# Reglas FAILED: patient_id nulo, fecha futura, código ICD-10 con formato incorrecto
failed_diag = [
    (col("patient_id").isNull(),                        "patient_id_nulo"),
    (col("visit_date") > current_date(),                "fecha_visita_futura"),
    # ICD-10 válido: letra seguida de al menos 2 dígitos (ej: A01, Z99.1)
    (regexp_extract(col("primary_icd10"), r'^[A-Z]\d{2}', 0) == "",
                                                        "icd10_formato_invalido"),
]

# No hay reglas QUARANTINE (es la tabla estructuralmente más limpia)
quarantine_diag = []  # Diagnoses es la tabla más limpia estructuralmente

# Aplicamos las reglas a diagnoses
df_diag_clean, r = aplicar_reglas(
    df_diagnoses, failed_diag, quarantine_diag, "diagnoses"
)
resumenes.append(r)
# Subida parcial de log tras esta seccion
_subir_log("calidad/calidad_resumen_final", STORAGE_ACCOUNT)


## 4. Resumen Final y Persistencia de Logs


In [ ]:
# ══════════════════════════════════════════════════════════════
# RESUMEN FINAL Y LOGS
# ══════════════════════════════════════════════════════════════
# ── RESUMEN GLOBAL ──
# Mostramos estadísticas por tabla en el logger y en pantalla
from pyspark.sql.functions import count

# ── Resumen global en logger ──
logger.info("")
logger.info("=" * 70)
logger.info("RESUMEN FINAL DE CALIDAD")
logger.info("=" * 70)
logger.info(f"{'Tabla':<20} {'Total':>8} {'Failed':>8} {'Quarantine':>12} {'Clean':>8} {'%Clean':>8}")
logger.info("-" * 70)
for r in resumenes:
    logger.info(
        f"{r['tabla']:<20} {r['total']:>8,} {r['failed']:>8,} "
        f"{r['quarantine']:>12,} {r['clean']:>8,} {r['pct_clean']:>7.2f}%"
    )

# Calculamos los totales globales para el resumen final
total_all     = sum(r['total']      for r in resumenes)
total_failed  = sum(r['failed']     for r in resumenes)
total_quar    = sum(r['quarantine'] for r in resumenes)
total_clean   = sum(r['clean']      for r in resumenes)
pct_clean_all = round(total_clean / total_all * 100, 2) if total_all > 0 else 0

logger.info("-" * 70)
logger.info(
    f"{'TOTAL':<20} {total_all:>8,} {total_failed:>8,} "
    f"{total_quar:>12,} {total_clean:>8,} {pct_clean_all:>7.2f}%"
)
logger.info("=" * 70)
logger.info(f"Carpeta failed:      {base_rejected}")
logger.info(f"Carpeta quarantine:  {base_quarantine}")

# ── INFORME DETALLADO POR TABLA ──
# Leemos los archivos failed/quarantine de cada tabla y mostramos desglose por motivo
# ── Informe visual por tabla y motivo ──
print()
print("=" * 70)
logger.info("INFORME DE CALIDAD DE DATOS — DETALLE DE RECHAZOS")
print("=" * 70)
print(f"\n{'Tabla':<20} {'Total':>8} {'Failed':>8} {'Quarantine':>12} {'Clean':>8} {'%Clean':>8}")
print("-" * 70)
for r in resumenes:
    print(f"{r['tabla']:<20} {r['total']:>8,} {r['failed']:>8,} "
          f"{r['quarantine']:>12,} {r['clean']:>8,} {r['pct_clean']:>7.2f}%")
print("-" * 70)
print(
    f"{'TOTAL':<20} {total_all:>8,} {total_failed:>8,} "
    f"{total_quar:>12,} {total_clean:>8,} {pct_clean_all:>7.2f}%"
)
print()

for tabla in ["patients", "outcomes", "lab_results", "medications", "diagnoses"]:
    print("=" * 70)
    logger.info(f"  TABLA: {tabla.upper()}")
    print("=" * 70)

    # -- FAILED --
    try:
        df_f = spark.read.parquet(f"{base_rejected}/{tabla}")
        n_f  = df_f.count()
        if n_f > 0:
            print(f"\n  🔴 FAILED ({n_f:,} registros) — desglose por motivo:")
            df_f.groupBy("motivo_rechazo") \
                .agg(count("*").alias("registros")) \
                .orderBy(col("registros").desc()) \
                .show(truncate=False)
        else:
            print(f"\n  🔴 FAILED: 0 registros\n")
    except Exception:
        print(f"\n  🔴 FAILED: 0 registros (carpeta no creada)\n")

    # -- QUARANTINE --
    try:
        df_q = spark.read.parquet(f"{base_quarantine}/{tabla}")
        n_q  = df_q.count()
        if n_q > 0:
            logger.info(f"  🟡 QUARANTINE ({n_q:,} registros) — desglose por motivo:")
            df_q.groupBy("motivo_alerta") \
                .agg(count("*").alias("registros")) \
                .orderBy(col("registros").desc()) \
                .show(truncate=False)
        else:
            print(f"  🟡 QUARANTINE: 0 registros\n")
    except Exception:
        print(f"  🟡 QUARANTINE: 0 registros (carpeta no creada)\n")

print("=" * 70)
logger.info("FIN DEL INFORME")
print("=" * 70)

# ── FINALIZACIÓN ──
# Cerramos el logger y subimos el log al Data Lake para trazabilidad
# ── Cerrar logger y guardar log en Data Lake ──
for handler in logging.root.handlers[:]:
    handler.flush()
    handler.close()
    logging.root.removeHandler(handler)

try:
    from notebookutils import mssparkutils
    fecha_str  = datetime.now().strftime('%Y%m%d_%H%M')
    ruta_log   = f"abfss://{CONTAINER_LOGS}@{STORAGE_ACCOUNT}.dfs.core.windows.net/calidad/calidad_{fecha_str}.log"
    ruta_local = "file://" + os.path.abspath(log_filename)
    mssparkutils.fs.cp(ruta_local, ruta_log)
    print(f"\nLog guardado en: {ruta_log}")
except Exception as e:
    logger.warning(f"No se pudo copiar el log: {e}")

logger.info("EXECUTION STATUS: SUCCESS")
_enviar_telegram("✅ FINALIZADO: 01B Calidad Bronze")
spark.stop()

# Plan de Validaciones de Calidad de Datos
**Notebook:** `01b_calidad_bronze_clean.ipynb`  
**Origen:** Capa Bronze — Parquet  
**Destino errores:** `medicalproyect-raw/failed/` · `medicalproyect-raw/quarantine/`

---

## Criterio de clasificación

| Nivel | Carpeta | Criterio |
|---|---|---|
| 🔴 **FAILED** | `/failed/` | Registro inválido de forma bloqueante. No puede continuar en el pipeline bajo ninguna circunstancia. |
| 🟡 **QUARANTINE** | `/quarantine/` | Registro estadísticamente sospechoso o anómalo, pero no imposible. Se aparta para revisión manual sin bloquear el pipeline. |
| 🟢 **CLEAN** | Continúa al Silver | Pasa todas las validaciones. |

---

## PATIENTS

| Columna | Regla | Destino |
|---|---|---|
| `patient_id` | Es nulo | 🔴 FAILED — `patient_id_nulo` |
| `age` | Es nulo, negativo o mayor de 120 | 🔴 FAILED — `age_invalida` |
| `diastolic_bp` / `systolic_bp` | `diastolic_bp >= systolic_bp` (imposible fisiológicamente) | 🔴 FAILED — `bp_diastolica_mayor_sistolica` |
| `bmi` | Fuera del rango [10, 80] | 🟡 QUARANTINE — `bmi_fuera_rango` |
| `systolic_bp` | Fuera del rango [50, 250] mmHg | 🟡 QUARANTINE — `systolic_bp_extrema` |
| `diastolic_bp` | Fuera del rango [30, 150] mmHg | 🟡 QUARANTINE — `diastolic_bp_extrema` |
| `heart_rate` | Fuera del rango [20, 250] ppm | 🟡 QUARANTINE — `heart_rate_extrema` |
| `temperature_f` | Fuera del rango [86, 113] °F (31–45 °C) | 🟡 QUARANTINE — `temperatura_extrema` |
| `charlson_index` | Valor negativo | 🟡 QUARANTINE — `charlson_negativo` |

---

## OUTCOMES

| Columna | Regla | Destino |
|---|---|---|
| `patient_id` | Es nulo | 🔴 FAILED — `patient_id_nulo` |
| `discharge_date` / `admission_date` | `discharge_date < admission_date` | 🔴 FAILED — `discharge_antes_admission` |
| `length_of_stay_days` | Valor negativo | 🔴 FAILED — `estancia_negativa` |
| `readmitted_30d` / `days_to_readmission` | `readmitted_30d = True` pero `days_to_readmission > 30` (inconsistencia lógica) | 🔴 FAILED — `readmitted_inconsistente` |
| `total_charges_usd` | Valor negativo | 🟡 QUARANTINE — `cargos_negativos` |
| `days_to_readmission` | Valor negativo | 🟡 QUARANTINE — `dias_readmision_negativos` |
| `length_of_stay_days` | Superior a 365 días | 🟡 QUARANTINE — `estancia_superior_1anio` |
| `icu_days` / `length_of_stay_days` | `icu_days > length_of_stay_days` | 🟡 QUARANTINE — `icu_days_mayor_estancia` |

---

## LAB\_RESULTS

| Columna | Regla | Destino |
|---|---|---|
| `patient_id` | Es nulo | 🔴 FAILED — `patient_id_nulo` |
| `test_date` | Fecha posterior a la fecha de ejecución (futura) | 🔴 FAILED — `fecha_futura` |
| `reference_high` / `reference_low` | `reference_high < reference_low` (rango invertido) | 🔴 FAILED — `rango_referencia_invertido` |
| `value` | Valor negativo (analíticas no pueden ser negativas) | 🟡 QUARANTINE — `valor_analitica_negativo` |
| `is_abnormal` / `value` | `is_abnormal = False` pero el valor está fuera del rango `[reference_low, reference_high]` | 🟡 QUARANTINE — `is_abnormal_inconsistente` |

---

## MEDICATIONS

| Columna | Regla | Destino |
|---|---|---|
| `patient_id` | Es nulo | 🔴 FAILED — `patient_id_nulo` |
| `dose` | Cero o negativa | 🔴 FAILED — `dosis_cero_o_negativa` |
| `start_date` | Fecha posterior a la fecha de ejecución (futura) | 🟡 QUARANTINE — `fecha_inicio_futura` |
| `duration_days` | Valor negativo | 🟡 QUARANTINE — `duracion_negativa` |
| `adherence_pct` | Fuera del rango [0, 100] | 🟡 QUARANTINE — `adherencia_fuera_rango` |

---

## DIAGNOSES

| Columna | Regla | Destino |
|---|---|---|
| `patient_id` | Es nulo | 🔴 FAILED — `patient_id_nulo` |
| `visit_date` | Fecha posterior a la fecha de ejecución (futura) | 🔴 FAILED — `fecha_visita_futura` |
| `primary_icd10` | No cumple el patrón ICD-10: letra mayúscula seguida de al menos 2 dígitos (ej: `A01`, `Z99.1`) | 🔴 FAILED — `icd10_formato_invalido` |

---

## Resumen de reglas

| Tabla | Reglas FAILED | Reglas QUARANTINE | Total |
|---|---|---|---|
| `patients` | 3 | 6 | 9 |
| `outcomes` | 4 | 4 | 8 |
| `lab_results` | 3 | 2 | 5 |
| `medications` | 2 | 3 | 5 |
| `diagnoses` | 3 | 0 | 3 |
| **TOTAL** | **15** | **15** | **30** |

---

## Campos adicionales en los registros rechazados

Todos los registros que van a `/failed/` o `/quarantine/` incluyen tres columnas extra añadidas por el pipeline:

| Campo | Descripción |
|---|---|
| `motivo_rechazo` / `motivo_alerta` | Código(s) del motivo separados por `\|` en caso de múltiples infracciones |
| `tabla_origen` | Nombre de la tabla de la que procede el registro |
| `ts_rechazo` / `ts_alerta` | Timestamp del momento en que fue clasificado |
